In [4]:
smallnum = .112341231341234

myint = int(1/smallnum)
print(myint)

8


In [2]:
import time
from datetime import timedelta

time_start = time.perf_counter()
for j in range(0, 100000000):
    foo =3
time_end = time.perf_counter()
run_time = time_end - time_start

print("total time:" + str(timedelta(seconds=run_time)))

total time:0:00:04.003333


In [12]:
import requests

def notify(msg):
    requests.post(
        "https://ntfy.sh/negh_MCMC_runs_04_23_86_07_14_20_9_10_23",
        data=msg.encode("utf-8"),
        timeout=5
    )

notify("hello world")

In [13]:

mystr = "poop" + "\n"
mystr += "me" + "\n"

print(mystr)

import numpy as np

ary = np.zeros(5)
print(ary)

print(ary[0:1])


x =3

print(int(x > 0))


x = 0
print(int(x == 0))
x = 5
print(int(x == 0))
x = .5
print(int(x == 0))

poop
me

[0. 0. 0. 0. 0.]
[0.]
1
1
0
0


In [17]:
import os
import smtplib
from email.message import EmailMessage

def message_me(subject,message):
    user = "nglattholtz@gmail.com"
    pw = "irvl thcq apqa fnvw"
    to = "8187235144@vtext.com"
    host = "smtp.gmail.com"
    port = "465"


    msg = EmailMessage()
    msg["From"] = user
    msg["To"] = to
    msg["Subject"] = subject
    msg.set_content(message)


    with smtplib.SMTP_SSL(host, port) as smtp:
        smtp.login(user, pw)
        smtp.send_message(msg)


In [6]:
import smtplib
from email.message import EmailMessage
import os

import os
os.environ["SMTP_USER"] = "nglattholtz@gmail.com"
os.environ["SMTP_PASS"] = "irvl thcq apqa fnvw" 
os.environ["SMS_GATEWAY_TO"] = "8187235144@vtext.com"

msg = EmailMessage()
msg["From"] = os.environ["SMTP_USER"]
msg["To"] = os.environ["SMTP_USER"]   # send to yourself first
msg["Subject"] = "SMTP test"
msg.set_content("If you see this, App Password works.")

with smtplib.SMTP_SSL("smtp.gmail.com", 465) as s:
    s.login(os.environ["SMTP_USER"], os.environ["SMTP_PASS"])
    s.send_message(msg)

print("Email sent successfully.")


Email sent successfully.


In [7]:
import os
import smtplib
from email.message import EmailMessage

def send_sms(message: str, subject: str = "Job done") -> None:
    host = os.environ["SMTP_HOST"]
    port = int(os.environ.get("SMTP_PORT", "465"))
    user = os.environ["SMTP_USER"]
    pw   = os.environ["SMTP_PASS"]
    to   = os.environ["SMS_GATEWAY_TO"]

    # Keep SMS short; many gateways truncate.
    message = message.strip()
    if len(message) > 300:
        message = message[:297] + "..."

    msg = EmailMessage()
    msg["From"] = user
    msg["To"] = to
    msg["Subject"] = subject
    msg.set_content(message)

    with smtplib.SMTP_SSL(host, port) as smtp:
        smtp.login(user, pw)
        smtp.send_message(msg)

## Model Problem B: Matrix Coefficents from Solutions

We next consider inverse problem of estimating the coefficents of an $n \times n$ anti-semetric matrix $A$ from data $\theta \in \mathbb{R}^n$ gathered according the measurement model:
$$
    z = \mathcal{O}(\theta(A)) + \eta \qquad \text{ where } \eta \sim N(0, \sigma^2)
    \tag{MM2}
$$
Here $\theta := \theta(A)$ is the solution of
$$
    (A + \kappa I)\theta = g
$$
and where $\mathcal{O}(\theta)$ defined by $\mathcal{O}: \mathbb{R}^n \to \mathbb{R}^m$ is a partial observation of $\theta$. Typically
$$
    \mathcal{O}((\theta_1, \ldots, \theta_n)) = (\theta_1,\ldots, \theta_m)
$$
Here the Bayesian posterior will take the form
$$
    \mu(dA) \propto \exp\left( - \frac{1}{2 \sigma^2} 
    | \mathcal{O}( (A -\kappa I)^{-1} g) - z|^2 
    - \frac{1}{2} <C^{-1}A, A>\right) dA.
    \tag{S1}
$$
which is a target in $d = n(n-1)/2$


In [4]:
#Set-up: Packages to Import and Untility Functions

#Core Numerical Packages for Paral
import MCMC_Sampliers as MCMCsmp

#Numerical Elements
from numpy.linalg import norm
import numpy as np
from numpy import dot, array, transpose, diag
import random
import math

#Input Output utils
import os
import pandas as pd

def writeCSV(filenm, sampArray, newFile = False):
    if newFile:
        pd.DataFrame(sampArray).to_csv(filenm, mode='w', index=False, header=False)
    else:
        pd.DataFrame(sampArray).to_csv(filenm, mode='a', index=False, header=False)

def readCSV(filenm):   
    df = pd.read_csv(filenm)
    MpCNsamp2 = df.to_numpy()
    return MpCNsamp2

#Fun Progress Bar
from tqdm.notebook import tqdm

#Parallel executation functions
from functools import partial
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp
import inspect
print("Current cpu count:" + str(mp.cpu_count()))

#Histogram and Other Graphics Utililities
import matplotlib.pyplot as plt
import matplotlib as mpl

def getComp(sampLst,indx):
    return [item[indx] for item in sampLst]

def makeHistGrid(R, dr, sampList, NumParams,saveLoc, hidePlt = True):
    SampLstsGd = [getComp(sampList,j) for j in range(0,NumParams)]
    numbins= int(2*R/dr)
    x_bins = np.linspace(-R, R, numbins) 
    y_bins = np.linspace(-R, R, numbins)

    fig, axs = plt.subplots(NumParams, NumParams,figsize=(15,15))
    for i in range(0,NumParams):
        for j in range(0,NumParams):
            if i == j:
                axs[i,j].hist(SampLstsGd[i], density=True, bins=x_bins)
            else:
                axs[i,j].hist2d(SampLstsGd[j],SampLstsGd[i],bins= [x_bins,y_bins])
        for ax in axs.flat:
            ax.label_outer()
    plt.savefig(saveLoc)
    if hidePlt:
        plt.close()

def generate_colors(n, cmap_name="tab10"):
    """
    Return a list of `n` hex colors taken at equal intervals
    from a Matplotlib colormap (default: 'tab10').

    Parameters
    ----------
    n : int
        Number of colors you need.
    cmap_name : str, optional
        Any valid Matplotlib colormap name.  Sequential maps
        ('viridis', 'plasma', …) or qualitative maps ('tab10',
        'Set3', …) both work.

    Returns
    -------
    list[str]
        Hex strings like '#1f77b4' ready for plotting.
    """
    # Get a ListedColormap with exactly n entries
    cmap = plt.get_cmap(cmap_name, n)
    # Convert RGBA to hex for convenience
    return [mpl.colors.to_hex(cmap(i)) for i in range(cmap.N)]



/Users/negh_macpro/Desktop/Git_Projects/MCMC_Code_Base/Large_p_Limit/MCMC_Sampliers.py:301: SyntaxWarning: invalid escape sequence '\i'
  """


Current cpu count:11


In [88]:
#Experiment 5: Set-up

#This experiment is based on Model Problem B

#We Specifying Problem Parameters as follows
#Model dimension and parameter space size are
ModDmEx5 = 4
NumParmsEx5 = int(ModDmEx5*(ModDmEx5 -1)/2)

#This time we set g = [1,...,1] and z = kap^{-1} g which is
#leads to a stiff log likihood of the form 
#Pot(A) = kap^{-1}||(A - kap I)^{-1} A||^2
kapEx5 = 0.01 #Make kap smaller to get a stiffer for A oriented problem problem
sigEx5 = 0.01 #Make sigma smaller for a likelihood dominated problem
cov0 = 9
gam = 1.1
CovDiag = [cov0* (j**(-gam)) for j in list(range(1,NumParmsEx5+1))]
#Q = MCMCsmp.rndm_orth_matrix(NumParmsEx4)
#CovEx4 = Q.T@ MCMCsmp.mkDiagCov(CovDiag)@ Q
CovEx5 = MCMCsmp.mkDiagCov(CovDiag)

def FindMatchFull(numParm, modDm, Cov, kap, tol, gam):
    curErr = 2
    Idd = np.eye(modDm)
    mnA = np.zeros(numParm)
    mnG = np.zeros(modDm)
    CovInv = np.linalg.inv(Cov)
    numtries = 1
    while curErr > tol:
        Acur = np.random.multivariate_normal(mnA, Cov,1)[0]
        Gcur = np.random.multivariate_normal(mnG, .1*Idd,1)[0].T
        Qcur = MCMCsmp.rndm_orth_matrix(numParm)
        tAcur = gam* Qcur.T @ Acur @ Qcur
        curAerr = np.linalg.norm(MCMCsmp.getThA(modDm, tAcur, Gcur, kap) - MCMCsmp.getThA(modDm, Acur, Gcur, kap)) 
        rotInBulk = max(0,np.dot(CovInv @ tAcur, tAcur) -np.dot(CovInv @ Acur, Acur))
        curErr = min(curAerr + rotInBulk,curErr)
        numtries = numtries+ 1
        if numtries % 100000 == 0:
            print("Indx: " + str(numtries))
            print("A error: " +str(curAerr))
            print("Expansion: "+ str(rotInBulk))
            print("Running Error" + str(curErr))
    return Acur, Gcur, tAcur, Qcur,numtries

A, g, tA, Q, n= FindMatchFull(NumParmsEx5, ModDmEx5, CovEx5, kapEx5, .002, .9)

print(A)
print(g)
print(tA)
print(Q)
print(n)

print(MCMCsmp.getThA(ModDmEx5, A, g, kapEx5))
print(MCMCsmp.getThA(ModDmEx5, tA, g, kapEx5))




Indx: 100000
A error: 1.735339748067597
Expansion: 1.287143829852603
Running Error0.00872191017959563
Indx: 200000
A error: 0.17551132636035524
Expansion: 2.9621844365956473
Running Error0.00872191017959563
Indx: 300000
A error: 0.3615684631216183
Expansion: 5.854165799573517
Running Error0.00872191017959563
Indx: 400000
A error: 2.02351373855442
Expansion: 5.26538072524769
Running Error0.00872191017959563
Indx: 500000
A error: 3.8468375376221418
Expansion: 0.7431208078375664
Running Error0.00872191017959563
Indx: 600000
A error: 1.121915540996486
Expansion: 4.669383606586059
Running Error0.00872191017959563
Indx: 700000
A error: 0.48871977608792794
Expansion: 2.695762824532949
Running Error0.008712696935129315
Indx: 800000
A error: 1.0857697001853759
Expansion: 6.353926201567347
Running Error0.004313690021655091
Indx: 900000
A error: 0.8544864711137624
Expansion: 0
Running Error0.004313690021655091
Indx: 1000000
A error: 1.0913445833354087
Expansion: 0.2455244702900854
Running Error0.

KeyboardInterrupt: 

In [79]:
#Make Problem Information File

zEx5 = MCMCsmp.getThA(ModDmEx5, A, g, kapEx5)
gvecEx5= g
dataDimEx5 = 3

expId =  16 #Experiment ID
FileNmBase = "Data/Large_p_study/Experiment_5/" + "Ex_ID_"+ str(expId) + "/"
os.makedirs(FileNmBase, exist_ok=True)

probDataFile = FileNmBase + "Problem_Info.txt"
with open(probDataFile, 'w') as file:
    file.write("Target Type:  Model Problem B \n")
    file.write("Model Dim: " + str(NumParmsEx5) + "\n")
    file.write("g: " + str(gvecEx5) + "\n")
    file.write("z: " + str(zEx5[ModDmEx5-1]) + "\n")
    file.write("sig: " + str(sigEx5)+ "\n")
    file.write("kappa: " + str(kapEx5)+ "\n")
    file.write("Prior Cov: " + str(CovEx5) + "\n")

#Make Loglikehood function with lambda workaround
#using PotExAD(a, gvec, sig, ModDm, z, kap, dataDim, mode = None):
#print(inspect.signature(MCMCsmp.PotExAD))


PotEx5Pass = partial(MCMCsmp.PotExAD, gvec=gvecEx5, sig=sigEx5, ModDm=ModDmEx5, z=zEx5, kap=kapEx5, dataDim=dataDimEx5, mode = "soft")

print(CovEx5)

[[9.         0.         0.         0.         0.         0.        ]
 [0.         4.19864846 0.         0.         0.         0.        ]
 [0.         0.         2.68787538 0.         0.         0.        ]
 [0.         0.         0.         1.95873877 0.         0.        ]
 [0.         0.         0.         0.         1.53241186 0.        ]
 [0.         0.         0.         0.         0.         1.2539382 ]]


In [80]:
#Experiment 5
#Perform a baseline/warm-up MCMC run

q0 = np.zeros(NumParmsEx5)
rhoWarm = .3
pWarm = 100

numSmpWarm = 5000
burninWarm = 4000

WarmSamps = MCMCsmp.MpCN(q0,NumParmsEx5,CovEx5,rhoWarm,PotEx5Pass,pWarm,numSmpWarm)

NumRuns =  1000 #of total runs
NumSamps = 1000 #samples per run


#NumRuns =  10 #of total runs
#NumSamps = 10000 #samples per run

if __name__ == "__main__": 
    
    with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
        MCMCsampRun = []
        for run in range(0,NumRuns):
            strIndx = random.randint(burninWarm, numSmpWarm)
            q0zCur = WarmSamps[strIndx]
            CurfNinput = (q0zCur,NumParmsEx5,CovEx5,rhoWarm,PotEx5Pass,pWarm,numSmpWarm)
            MCMCsampRun.append(pool.submit(MCMCsmp.MpCN,*CurfNinput))


        print("Total MCMC Runs: " + str(len(MCMCsampRun)))
        results = [None]*len(MCMCsampRun)
        # map Future -> index so we can keep ordered results
        where = {f:i for i, f in enumerate(MCMCsampRun)}

        for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
            i = where[f]
            results[i] = f.result()
    csvFileNm = FileNmBase + "Baseline_Samples.csv"
    for curRnInx in tqdm(range(0,len(MCMCsampRun)), desc= "Compiling Data"):
         writeCSV(csvFileNm,MCMCsampRun[curRnInx].result())

#Generate Histogram
#Dimensions For Histogram Plot

ex5MCMCSmpBase = readCSV(csvFileNm)
MCMCpoolLen = len(ex5MCMCSmpBase)
print("Number of MpCN Samples Now Available: " + str(MCMCpoolLen))

R = 7
dr = .1

histFileNm = FileNmBase + "Baseline_Histogram.png"
makeHistGrid(R, dr, ex5MCMCSmpBase, NumParmsEx5,histFileNm, True)

Total MCMC Runs: 1000


Parallel MCMC Runs:   0%|          | 0/1000 [00:00<?, ?it/s]

Compiling Data:   0%|          | 0/1000 [00:00<?, ?it/s]

Number of MpCN Samples Now Available: 5000999


In [84]:
##Generate rotations of the posterior for visualization

os.makedirs(FileNmBase + "/Rotations", exist_ok=True)

numRots = 5

for Rot_indx in tqdm(range(0,numRots)):

    RanRot = MCMCsmp.rndm_orth_matrix(NumParmsEx5)

    ex4MCMCSmpBaseRot = [RanRot @ A for A in ex5MCMCSmpBase]

    #Generate Histogram
    #Dimensions For Histogram Plot


    R = 5
    dr = .1

    histFileNm = FileNmBase + "/Rotations/" +"Baseline_Histogram_Rot_" + str(Rot_indx) + ".png"
    makeHistGrid(R, dr, ex4MCMCSmpBaseRot, NumParmsEx5,histFileNm, True)

  0%|          | 0/5 [00:00<?, ?it/s]

In [91]:
#Experiment 5: Set-up

#This experiment is based on Model Problem B

#We Specifying Problem Parameters as follows
#Model dimension and parameter space size are
ModDmEx5 = 4
NumParmsEx5 = int(ModDmEx5*(ModDmEx5 -1)/2)

#This time we set g = [1,...,1] and z = kap^{-1} g which is
#leads to a stiff log likihood of the form 
#Pot(A) = kap^{-1}||(A - kap I)^{-1} A||^2
kapEx5 = 0.01 #Make kap smaller to get a stiffer for A oriented problem problem
sigEx5 = 0.01 #Make sigma smaller for a likelihood dominated problem
cov0 = 9


gam = 1.1
CovDiag = [cov0* (j**(-gam)) for j in list(range(1,NumParmsEx5+1))]
#Q = MCMCsmp.rndm_orth_matrix(NumParmsEx4)
#CovEx4 = Q.T@ MCMCsmp.mkDiagCov(CovDiag)@ Q
CovEx5 = MCMCsmp.mkDiagCov(CovDiag)

def FindMatchFull(numParm, modDm, datacomp, Cov, kap, tol, lam):
    curErr = 2
    Idd = np.eye(modDm)
    mnA = np.zeros(numParm)
    mnG = np.zeros(modDm)
    CovInv = np.linalg.inv(Cov)
    numtries = 1
    while curErr > tol:
        Acur = np.random.multivariate_normal(mnA, Cov,1)[0]
        Gcur = np.random.multivariate_normal(mnG, .1*Idd,1)[0].T
        Qcur = MCMCsmp.rndm_orth_matrix(numParm)
        tAcur = lam* Qcur.T @ Acur @ Qcur
        curAerr = np.linalg.norm((MCMCsmp.getThA(modDm, tAcur, Gcur, kap) - MCMCsmp.getThA(modDm, Acur, Gcur, kap))*datacomp) 
        rotInBulk = max(0,np.dot(CovInv @ tAcur, tAcur) -np.dot(CovInv @ Acur, Acur))
        curErr = min(curAerr + rotInBulk,curErr)
        numtries = numtries+ 1
        if numtries % 100000 == 0:
            print("Indx: " + str(numtries))
            print("A error: " +str(curAerr))
            print("Expansion: "+ str(rotInBulk))
            print("Running Error" + str(curErr))
    return Acur, Gcur, tAcur, Qcur,numtries

datacomp2nd4th = np.array([1,0,0,1])

Atru, gtru, tA, Q, n= FindMatchFull(NumParmsEx5, ModDmEx5, datacomp2nd4th, CovEx5, kapEx5, .0005, .9)

print(Atru)
print(gtru)
print(tA)
print(Q)
print(n)

print(MCMCsmp.getThA(ModDmEx5, Atru, gtru, kapEx5)*datacomp2nd4th)
print(MCMCsmp.getThA(ModDmEx5, tA, gtru, kapEx5)*datacomp2nd4th)



Indx: 100000
A error: 11.221127438036861
Expansion: 0
Running Error0.0012675833476912142
Indx: 200000
A error: 0.2668999355996978
Expansion: 0.5276300319265648
Running Error0.0012675833476912142
Indx: 300000
A error: 4.989787251512229
Expansion: 11.40326353913724
Running Error0.0012675833476912142
Indx: 400000
A error: 0.5063841327842251
Expansion: 0
Running Error0.0008990237186629092
Indx: 500000
A error: 0.21810968835889383
Expansion: 2.1608770073166443
Running Error0.0008990237186629092
Indx: 600000
A error: 0.38299376553495923
Expansion: 1.582937823272065
Running Error0.0006324587020905945
Indx: 700000
A error: 0.2948369582629604
Expansion: 0.7506905481026616
Running Error0.0006324587020905945
Indx: 800000
A error: 0.6171592029465807
Expansion: 0.5896797619879255
Running Error0.0006324587020905945
Indx: 900000
A error: 0.18791979609117168
Expansion: 0.4043832774756533
Running Error0.0006324587020905945
Indx: 1000000
A error: 0.49751361985981624
Expansion: 0
Running Error0.000632458

In [7]:
#Experiment 5: Set-up

#This experiment is based on Model Problem B

#We Specifying Problem Parameters as follows
#Model dimension and parameter space size are
ModDmEx5 = 4
NumParmsEx5 = int(ModDmEx5*(ModDmEx5 -1)/2)

#This time we set g = [1,...,1] and z = kap^{-1} g which is
#leads to a stiff log likihood of the form 
#Pot(A) = kap^{-1}||(A - kap I)^{-1} A||^2
kapEx5 = 0.01 #Make kap smaller to get a stiffer for A oriented problem problem
sigEx5 = 0.01 #Make sigma smaller for a likelihood dominated problem
cov0 = 9


gam = 1.1
CovDiag = [cov0* (j**(-gam)) for j in list(range(1,NumParmsEx5+1))]
#Q = MCMCsmp.rndm_orth_matrix(NumParmsEx4)
#CovEx4 = Q.T@ MCMCsmp.mkDiagCov(CovDiag)@ Q
CovEx5 = MCMCsmp.mkDiagCov(CovDiag)

datacomp2nd4th = np.array([1,0,0,1])


Atru = np.array([ 3.51472678, -3.89954437,  0.41883695, -1.23214839, -2.0195991,  0.04775199])
gvecEx5= np.array([-0.31350811, -0.04480832,  0.070213,   -0.04616241])


zEx5 = MCMCsmp.getThA(ModDmEx5, Atru, gvecEx5, kapEx5)
print(zEx5)



expId =  17 #Experiment ID
FileNmBase = "Data/Large_p_study/Experiment_5/" + "Ex_ID_"+ str(expId) + "/"
os.makedirs(FileNmBase, exist_ok=True)

probDataFile = FileNmBase + "Problem_Info.txt"
with open(probDataFile, 'w') as file:
    file.write("Target Type:  Model Problem B \n")
    file.write("Model Dim: " + str(NumParmsEx5) + "\n")
    file.write("g: " + str(gvecEx5) + "\n")
    file.write("z: " + str(zEx5*datacomp2nd4th) + "\n")
    file.write("sig: " + str(sigEx5)+ "\n")
    file.write("kappa: " + str(kapEx5)+ "\n")
    file.write("Prior Cov: " + str(CovEx5) + "\n")

#Make Loglikehood function with lambda workaround
#using PotExAD(a, gvec, sig, ModDm, z, kap, dataDim, mode = None):
#print(inspect.signature(MCMCsmp.PotExAD))


dataDimEx5 = 1

PotEx5Pass = partial(MCMCsmp.PotExAD_comp, gvec=gvecEx5, sig=sigEx5, ModDm=ModDmEx5, z=zEx5, kap=kapEx5, dataDim=dataDimEx5, data_comp = datacomp2nd4th, mode = "soft")

print(CovEx5)

[ 0.02366996 -0.01625845  0.05983202 -0.05559005]
[[9.         0.         0.         0.         0.         0.        ]
 [0.         4.19864846 0.         0.         0.         0.        ]
 [0.         0.         2.68787538 0.         0.         0.        ]
 [0.         0.         0.         1.95873877 0.         0.        ]
 [0.         0.         0.         0.         1.53241186 0.        ]
 [0.         0.         0.         0.         0.         1.2539382 ]]


In [8]:
#Experiment 5
#Perform a baseline/warm-up MCMC run

q0 = np.zeros(NumParmsEx5)
rhoWarm = .3
pWarm = 100

numSmpWarm = 5000
burninWarm = 4000

WarmSamps = MCMCsmp.MpCN(q0,NumParmsEx5,CovEx5,rhoWarm,PotEx5Pass,pWarm,numSmpWarm)

NumRuns =  100 #of total runs
NumSamps = 1000 #samples per run


#NumRuns =  10 #of total runs
#NumSamps = 10000 #samples per run

if __name__ == "__main__": 
    
    with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
        MCMCsampRun = []
        for run in range(0,NumRuns):
            strIndx = random.randint(burninWarm, numSmpWarm)
            q0zCur = WarmSamps[strIndx]
            CurfNinput = (q0zCur,NumParmsEx5,CovEx5,rhoWarm,PotEx5Pass,pWarm,numSmpWarm)
            MCMCsampRun.append(pool.submit(MCMCsmp.MpCN,*CurfNinput))


        print("Total MCMC Runs: " + str(len(MCMCsampRun)))
        results = [None]*len(MCMCsampRun)
        # map Future -> index so we can keep ordered results
        where = {f:i for i, f in enumerate(MCMCsampRun)}

        for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
            i = where[f]
            results[i] = f.result()
    csvFileNm = FileNmBase + "Baseline_Samples.csv"
    for curRnInx in tqdm(range(0,len(MCMCsampRun)), desc= "Compiling Data"):
         writeCSV(csvFileNm,MCMCsampRun[curRnInx].result())

#Generate Histogram
#Dimensions For Histogram Plot

ex5MCMCSmpBase = readCSV(csvFileNm)
MCMCpoolLen = len(ex5MCMCSmpBase)
print("Number of MpCN Samples Now Available: " + str(MCMCpoolLen))

R = 7
dr = .1

histFileNm = FileNmBase + "Baseline_Histogram.png"
makeHistGrid(R, dr, ex5MCMCSmpBase, NumParmsEx5,histFileNm, True)

Total MCMC Runs: 100


Parallel MCMC Runs:   0%|          | 0/100 [00:00<?, ?it/s]

Compiling Data:   0%|          | 0/100 [00:00<?, ?it/s]

Number of MpCN Samples Now Available: 500099


In [9]:
##Generate rotations of the posterior for visualization

os.makedirs(FileNmBase + "/Rotations", exist_ok=True)

numRots = 10

for Rot_indx in tqdm(range(0,numRots)):

    RanRot = MCMCsmp.rndm_orth_matrix(NumParmsEx5)

    ex4MCMCSmpBaseRot = [RanRot @ A for A in ex5MCMCSmpBase]

    #Generate Histogram
    #Dimensions For Histogram Plot


    R = 5
    dr = .1

    histFileNm = FileNmBase + "/Rotations/" +"Baseline_Histogram_Rot_" + str(Rot_indx) + ".png"
    makeHistGrid(R, dr, ex4MCMCSmpBaseRot, NumParmsEx5,histFileNm, True)

  0%|          | 0/10 [00:00<?, ?it/s]

In [15]:
#Experiment 5: Set-up

#This experiment is based on Model Problem B

#We Specifying Problem Parameters as follows
#Model dimension and parameter space size are
ModDmEx5 = 7
NumParmsEx5 = int(ModDmEx5*(ModDmEx5 -1)/2)


#Pot(A) = kap^{-1}||(A - kap I)^{-1} A||^2
kapEx5 = 0.01 #Make kap smaller to get a stiffer for A oriented problem problem
sigEx5 = 0.01 #Make sigma smaller for a likelihood dominated problem
cov0 = 9


gam = 1.1
CovDiag = [cov0* (j**(-gam)) for j in list(range(1,NumParmsEx5+1))]
#Q = MCMCsmp.rndm_orth_matrix(NumParmsEx4)
#CovEx4 = Q.T@ MCMCsmp.mkDiagCov(CovDiag)@ Q
CovEx5 = MCMCsmp.mkDiagCov(CovDiag)

def FindMatchFull(numParm, modDm, datacomp, Cov, kap, tol, lam):
    curErr = 2
    IddGo = MCMCsmp.mkDiagCov([1,1,0,0,0,0,0])
    mnA = np.zeros(numParm)
    mnGo = np.zeros(modDm)
    CovInv = np.linalg.inv(Cov)
    numtries = 1
    while curErr > tol:
        Acur = np.random.multivariate_normal(mnA, Cov,1)[0]
        Gcur = np.random.multivariate_normal(mnGo, IddGo,1)[0]
        Qcur = MCMCsmp.rndm_orth_matrix(numParm)
        tAcur = lam* Qcur.T @ Acur @ Qcur
        curAerr = np.linalg.norm((MCMCsmp.getThA(modDm, tAcur, Gcur, kap) - MCMCsmp.getThA(modDm, Acur, Gcur, kap))*datacomp) 
        rotInBulk = max(0,np.dot(CovInv @ tAcur, tAcur) -np.dot(CovInv @ Acur, Acur))
        curErr = min(curAerr + rotInBulk,curErr)
        numtries = numtries+ 1
        if numtries % 100000 == 0:
            print("Indx: " + str(numtries))
            print("A error: " +str(curAerr))
            print("Expansion: "+ str(rotInBulk))
            print("Running Error" + str(curErr))
    return Acur, Gcur, tAcur, Qcur,numtries

datacomp2nd4th = np.array([1,1,0,0,0,0,0])

Atru, gtru, tA, Q, n= FindMatchFull(NumParmsEx5, ModDmEx5, datacomp2nd4th, CovEx5, kapEx5, .005, .9)

print(Atru)
print(gtru)
print(tA)
print(Q)
print(n)

print(MCMCsmp.getThA(ModDmEx5, Atru, gtru, kapEx5)*datacomp2nd4th)
print(MCMCsmp.getThA(ModDmEx5, tA, gtru, kapEx5)*datacomp2nd4th)



Indx: 100000
A error: 1.688594159393697
Expansion: 13.040681125786456
Running Error0.012697069696968568
Indx: 200000
A error: 1.9850631258076663
Expansion: 41.90339477450614
Running Error0.012008050086274289
[ 0.57541922  0.14051753 -1.52575094 -1.14934654 -0.4032132  -0.37587489
  0.93001742 -0.26616846  0.27357021  0.41851674 -1.08336748  0.47446799
  0.35270337 -0.20020011 -0.04149203 -0.46041394  1.23549751  0.76713626
  0.02046856  0.48446855 -1.20186114]
[-0.1811877  -0.10899281  0.          0.          0.          0.
  0.        ]
[-0.38493169  0.53178109  0.76421639 -0.62534299 -0.1538143  -0.03773151
 -1.57381347 -0.36633036 -0.42100691  0.03133543 -0.97643158  0.38910707
 -0.24115609  0.31853845 -0.39640374  0.63486228 -0.15292982  0.41174596
  0.73115146  0.82369555 -1.22813269]
[[-8.48123974e-02  1.06440052e-01  4.72705799e-01  8.08276622e-02
   4.24609138e-03 -1.77423720e-01  5.60174019e-02 -2.05059310e-01
  -2.74366801e-01  1.12348792e-01 -3.48400726e-01 -1.38071747e-01
 

In [24]:
#Experiment 5: Set-up

#This experiment is based on Model Problem B

#We Specifying Problem Parameters as follows
#Model dimension and parameter space size are
ModDmEx5 = 7
NumParmsEx5 = int(ModDmEx5*(ModDmEx5 -1)/2)

#This time we set g = [1,...,1] and z = kap^{-1} g which is
#leads to a stiff log likihood of the form 
#Pot(A) = kap^{-1}||(A - kap I)^{-1} A||^2
kapEx5 = 0.01 #Make kap smaller to get a stiffer for A oriented problem problem
sigEx5 = 0.01 #Make sigma smaller for a likelihood dominated problem
cov0 = 9


gam = 1.1
CovDiag = [cov0* (j**(-gam)) for j in list(range(1,NumParmsEx5+1))]
#Q = MCMCsmp.rndm_orth_matrix(NumParmsEx4)
#CovEx4 = Q.T@ MCMCsmp.mkDiagCov(CovDiag)@ Q
CovEx5 = MCMCsmp.mkDiagCov(CovDiag)

datacomp2nd4th = np.array([1,1,0,0,0,0,0])


#Atru = np.array([ 3.51472678, -3.89954437,  0.41883695, -1.23214839, -2.0195991,  0.04775199])
#gvecEx5= np.array([-0.31350811, -0.04480832,  0.070213,   -0.04616241])
Atru = np.array([0.57541922,0.14051753,-1.52575094,-1.14934654,-0.4032132,-0.37587489,
    0.93001742,-0.26616846,0.27357021,0.41851674,-1.08336748,0.47446799,
    0.35270337,-0.20020011,-0.04149203,-0.46041394,1.23549751,0.76713626,
    0.02046856,0.48446855,-1.20186114])
gvecEx5 = np.array([-0.1811877,-0.10899281,0,0,0,0,0])

#print(MCMCsmp.getThA(ModDmEx5, Atru, gtru, kapEx5)*datacomp2nd4th)
#print(MCMCsmp.getThA(ModDmEx5, tA, gtru, kapEx5)*datacomp2nd4th)
print(MCMCsmp.getThA(ModDmEx5, Atru, gvecEx5, kapEx5)*datacomp2nd4th)
print(MCMCsmp.getThA(ModDmEx5, tA, gvecEx5, kapEx5)*datacomp2nd4th)


zEx5 = MCMCsmp.getThA(ModDmEx5, Atru, gvecEx5, kapEx5)
print(zEx5)



expId =  18 #Experiment ID
FileNmBase = "Data/Large_p_study/Experiment_5/" + "Ex_ID_"+ str(expId) + "/"
os.makedirs(FileNmBase, exist_ok=True)

probDataFile = FileNmBase + "Problem_Info.txt"
with open(probDataFile, 'w') as file:
    file.write("Target Type:  Model Problem B \n")
    file.write("Model Dim: " + str(NumParmsEx5) + "\n")
    file.write("g: " + str(gvecEx5) + "\n")
    file.write("z: " + str(zEx5*datacomp2nd4th) + "\n")
    file.write("sig: " + str(sigEx5)+ "\n")
    file.write("kappa: " + str(kapEx5)+ "\n")
    file.write("Prior Cov: " + str(CovEx5) + "\n")

#Make Loglikehood function with lambda workaround
#using PotExAD(a, gvec, sig, ModDm, z, kap, dataDim, mode = None):
#print(inspect.signature(MCMCsmp.PotExAD))


dataDimEx5 = 2

PotEx5Pass = partial(MCMCsmp.PotExAD_comp, gvec=gvecEx5, sig=sigEx5, ModDm=ModDmEx5, z=zEx5, kap=kapEx5, dataDim=dataDimEx5, data_comp = datacomp2nd4th, mode = "soft")

[-0.06014185  0.07699557 -0.          0.          0.          0.
 -0.        ]
[-0.05945324  0.07771183  0.         -0.          0.         -0.
 -0.        ]
[-0.06014185  0.07699557 -0.42276153  0.0926916   0.07183851  0.05389794
 -0.21347102]


In [25]:
#Experiment 5
#Perform a baseline/warm-up MCMC run

q0 = np.zeros(NumParmsEx5)
rhoWarm = .3
pWarm = 100

numSmpWarm = 5000
burninWarm = 4000

WarmSamps = MCMCsmp.MpCN(q0,NumParmsEx5,CovEx5,rhoWarm,PotEx5Pass,pWarm,numSmpWarm)

NumRuns =  100 #of total runs
NumSamps = 1000 #samples per run


#NumRuns =  10 #of total runs
#NumSamps = 10000 #samples per run

if __name__ == "__main__": 
    
    with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
        MCMCsampRun = []
        for run in range(0,NumRuns):
            strIndx = random.randint(burninWarm, numSmpWarm)
            q0zCur = WarmSamps[strIndx]
            CurfNinput = (q0zCur,NumParmsEx5,CovEx5,rhoWarm,PotEx5Pass,pWarm,numSmpWarm)
            MCMCsampRun.append(pool.submit(MCMCsmp.MpCN,*CurfNinput))


        print("Total MCMC Runs: " + str(len(MCMCsampRun)))
        results = [None]*len(MCMCsampRun)
        # map Future -> index so we can keep ordered results
        where = {f:i for i, f in enumerate(MCMCsampRun)}

        for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
            i = where[f]
            results[i] = f.result()
    csvFileNm = FileNmBase + "Baseline_Samples.csv"
    for curRnInx in tqdm(range(0,len(MCMCsampRun)), desc= "Compiling Data"):
         writeCSV(csvFileNm,MCMCsampRun[curRnInx].result())

#Generate Histogram
#Dimensions For Histogram Plot

ex5MCMCSmpBase = readCSV(csvFileNm)
MCMCpoolLen = len(ex5MCMCSmpBase)
print("Number of MpCN Samples Now Available: " + str(MCMCpoolLen))

R = 7
dr = .1

histFileNm = FileNmBase + "Baseline_Histogram.png"
makeHistGrid(R, dr, ex5MCMCSmpBase, NumParmsEx5,histFileNm, True)

Total MCMC Runs: 100


Parallel MCMC Runs:   0%|          | 0/100 [00:00<?, ?it/s]

Compiling Data:   0%|          | 0/100 [00:00<?, ?it/s]

Number of MpCN Samples Now Available: 500099


In [27]:
##Generate rotations of the posterior for visualization

os.makedirs(FileNmBase + "/Rotations", exist_ok=True)

numRots = 1

for Rot_indx in tqdm(range(0,numRots)):

    RanRot = MCMCsmp.rndm_orth_matrix(NumParmsEx5)

    ex4MCMCSmpBaseRot = [RanRot @ A for A in ex5MCMCSmpBase]

    #Generate Histogram
    #Dimensions For Histogram Plot


    R = 5
    dr = .1

    histFileNm = FileNmBase + "/Rotations/" +"Baseline_Histogram_Rot_" + str(Rot_indx) + ".png"
    makeHistGrid(R, dr, ex4MCMCSmpBaseRot, NumParmsEx5,histFileNm, True)

  0%|          | 0/1 [00:00<?, ?it/s]

In [29]:
#Experiment 5: Set-up

#This experiment is based on Model Problem B

#We Specifying Problem Parameters as follows
#Model dimension and parameter space size are
ModDmEx5 = 7
NumParmsEx5 = int(ModDmEx5*(ModDmEx5 -1)/2)


#Pot(A) = kap^{-1}||(A - kap I)^{-1} A||^2
kapEx5 = 0.01 #Make kap smaller to get a stiffer for A oriented problem problem
sigEx5 = 0.01 #Make sigma smaller for a likelihood dominated problem
cov0 = 9


gam = 1.1
CovDiag = [cov0* (j**(-gam)) for j in list(range(1,NumParmsEx5+1))]
#Q = MCMCsmp.rndm_orth_matrix(NumParmsEx4)
#CovEx4 = Q.T@ MCMCsmp.mkDiagCov(CovDiag)@ Q
CovEx5 = MCMCsmp.mkDiagCov(CovDiag)

def FindMatchFull(numParm, modDm, datacomp, Cov, kap, tol, lam):
    curErr = 2
    IddGo = MCMCsmp.mkDiagCov([1,1,0,0,0,0,0])
    mnA = np.zeros(numParm)
    mnGo = np.zeros(modDm)
    CovInv = np.linalg.inv(Cov)
    numtries = 1
    while curErr > tol:
        Acur = np.random.multivariate_normal(mnA, Cov,1)[0]
        Gcur = np.random.multivariate_normal(mnGo, IddGo,1)[0]
        #Qcur = MCMCsmp.rndm_orth_matrix(numParm)
        tAcur = -1*lam* Acur
        curAerr = np.linalg.norm((MCMCsmp.getThA(modDm, tAcur, Gcur, kap) - MCMCsmp.getThA(modDm, Acur, Gcur, kap))*datacomp) 
        rotInBulk = max(0,np.dot(CovInv @ tAcur, tAcur) -np.dot(CovInv @ Acur, Acur))
        curErr = min(curAerr + rotInBulk,curErr)
        numtries = numtries+ 1
        if numtries % 100000 == 0:
            print("Indx: " + str(numtries))
            print("A error: " +str(curAerr))
            print("Expansion: "+ str(rotInBulk))
            print("Running Error" + str(curErr))
    return Acur, Gcur, tAcur, numtries

datacomp2nd4th = np.array([1,1,0,0,0,0,0])

Atru, gtru, tA, n= FindMatchFull(NumParmsEx5, ModDmEx5, datacomp2nd4th, CovEx5, kapEx5, .00001, .9)

print(Atru)
print(gtru)
print(tA)
#print(Q)
print(n)

print(MCMCsmp.getThA(ModDmEx5, Atru, gtru, kapEx5)*datacomp2nd4th)
print(MCMCsmp.getThA(ModDmEx5, tA, gtru, kapEx5)*datacomp2nd4th)


Indx: 100000
A error: 0.0398526796287393
Expansion: 0
Running Error3.3899402792750687e-05
Indx: 200000
A error: 0.4453878000853706
Expansion: 0
Running Error3.3899402792750687e-05
Indx: 300000
A error: 0.20983421140030373
Expansion: 0
Running Error1.4437200438369696e-05
[-0.92879947  1.48474133  0.89270313 -0.41366859  2.51797     0.92927613
  0.08269988  2.53407909 -1.66756712 -0.82445637 -0.56299822 -0.24828998
  1.13590782 -0.39862417 -0.21290237 -0.30806064  0.91146858 -0.93936571
  1.28909809 -0.05700955 -0.05742663]
[-0.01604909 -0.0141896   0.          0.          0.          0.
  0.        ]
[ 0.83591952 -1.3362672  -0.80343281  0.37230173 -2.266173   -0.83634852
 -0.07442989 -2.28067118  1.50081041  0.74201073  0.5066984   0.22346098
 -1.02231704  0.35876175  0.19161214  0.27725458 -0.82032172  0.84542914
 -1.16018829  0.05130859  0.05168396]
352209
[-0.12385033 -0.18916701 -0.          0.         -0.         -0.
  0.        ]
[-0.12385354 -0.1891699  -0.          0.         -

In [30]:
#Experiment 5: Set-up

#This experiment is based on Model Problem B

#We Specifying Problem Parameters as follows
#Model dimension and parameter space size are
ModDmEx5 = 7
NumParmsEx5 = int(ModDmEx5*(ModDmEx5 -1)/2)

#This time we set g = [1,...,1] and z = kap^{-1} g which is
#leads to a stiff log likihood of the form 
#Pot(A) = kap^{-1}||(A - kap I)^{-1} A||^2
kapEx5 = 0.01 #Make kap smaller to get a stiffer for A oriented problem problem
sigEx5 = 0.01 #Make sigma smaller for a likelihood dominated problem
cov0 = 9


gam = 1.1
CovDiag = [cov0* (j**(-gam)) for j in list(range(1,NumParmsEx5+1))]
#Q = MCMCsmp.rndm_orth_matrix(NumParmsEx4)
#CovEx4 = Q.T@ MCMCsmp.mkDiagCov(CovDiag)@ Q
CovEx5 = MCMCsmp.mkDiagCov(CovDiag)

datacomp2nd4th = np.array([1,1,0,0,0,0,0])


#Atru = np.array([ 3.51472678, -3.89954437,  0.41883695, -1.23214839, -2.0195991,  0.04775199])
#gvecEx5= np.array([-0.31350811, -0.04480832,  0.070213,   -0.04616241])
gvecEx5 = gtru

#print(MCMCsmp.getThA(ModDmEx5, Atru, gtru, kapEx5)*datacomp2nd4th)
#print(MCMCsmp.getThA(ModDmEx5, tA, gtru, kapEx5)*datacomp2nd4th)
print(MCMCsmp.getThA(ModDmEx5, Atru, gvecEx5, kapEx5)*datacomp2nd4th)
print(MCMCsmp.getThA(ModDmEx5, tA, gvecEx5, kapEx5)*datacomp2nd4th)


zEx5 = MCMCsmp.getThA(ModDmEx5, Atru, gvecEx5, kapEx5)
print(zEx5)



expId =  19 #Experiment ID
FileNmBase = "Data/Large_p_study/Experiment_5/" + "Ex_ID_"+ str(expId) + "/"
os.makedirs(FileNmBase, exist_ok=True)

probDataFile = FileNmBase + "Problem_Info.txt"
with open(probDataFile, 'w') as file:
    file.write("Target Type:  Model Problem B \n")
    file.write("Model Dim: " + str(NumParmsEx5) + "\n")
    file.write("g: " + str(gvecEx5) + "\n")
    file.write("z: " + str(zEx5*datacomp2nd4th) + "\n")
    file.write("sig: " + str(sigEx5)+ "\n")
    file.write("kappa: " + str(kapEx5)+ "\n")
    file.write("Prior Cov: " + str(CovEx5) + "\n")

#Make Loglikehood function with lambda workaround
#using PotExAD(a, gvec, sig, ModDm, z, kap, dataDim, mode = None):
#print(inspect.signature(MCMCsmp.PotExAD))


dataDimEx5 = 2

PotEx5Pass = partial(MCMCsmp.PotExAD_comp, gvec=gvecEx5, sig=sigEx5, ModDm=ModDmEx5, z=zEx5, kap=kapEx5, dataDim=dataDimEx5, data_comp = datacomp2nd4th, mode = "soft")

[-0.12385033 -0.18916701 -0.          0.         -0.         -0.
  0.        ]
[-0.12385354 -0.1891699  -0.          0.         -0.         -0.
  0.        ]
[-0.12385033 -0.18916701 -0.39866036  0.08512582 -0.08479351 -0.06492541
  0.48834834]


In [31]:
#Experiment 5
#Perform a baseline/warm-up MCMC run

q0 = np.zeros(NumParmsEx5)
rhoWarm = .3
pWarm = 100

numSmpWarm = 5000
burninWarm = 4000

WarmSamps = MCMCsmp.MpCN(q0,NumParmsEx5,CovEx5,rhoWarm,PotEx5Pass,pWarm,numSmpWarm)

NumRuns =  100 #of total runs
NumSamps = 1000 #samples per run


#NumRuns =  10 #of total runs
#NumSamps = 10000 #samples per run

if __name__ == "__main__": 
    
    with ProcessPoolExecutor(max_workers=mp.cpu_count()) as pool:
        MCMCsampRun = []
        for run in range(0,NumRuns):
            strIndx = random.randint(burninWarm, numSmpWarm)
            q0zCur = WarmSamps[strIndx]
            CurfNinput = (q0zCur,NumParmsEx5,CovEx5,rhoWarm,PotEx5Pass,pWarm,numSmpWarm)
            MCMCsampRun.append(pool.submit(MCMCsmp.MpCN,*CurfNinput))


        print("Total MCMC Runs: " + str(len(MCMCsampRun)))
        results = [None]*len(MCMCsampRun)
        # map Future -> index so we can keep ordered results
        where = {f:i for i, f in enumerate(MCMCsampRun)}

        for f in tqdm(as_completed(MCMCsampRun), total=len(MCMCsampRun), desc="Parallel MCMC Runs"):
            i = where[f]
            results[i] = f.result()
    csvFileNm = FileNmBase + "Baseline_Samples.csv"
    for curRnInx in tqdm(range(0,len(MCMCsampRun)), desc= "Compiling Data"):
         writeCSV(csvFileNm,MCMCsampRun[curRnInx].result())

#Generate Histogram
#Dimensions For Histogram Plot

ex5MCMCSmpBase = readCSV(csvFileNm)
MCMCpoolLen = len(ex5MCMCSmpBase)
print("Number of MpCN Samples Now Available: " + str(MCMCpoolLen))

R = 7
dr = .1

histFileNm = FileNmBase + "Baseline_Histogram.png"
makeHistGrid(R, dr, ex5MCMCSmpBase, NumParmsEx5,histFileNm, True)

Total MCMC Runs: 100


Parallel MCMC Runs:   0%|          | 0/100 [00:00<?, ?it/s]

Compiling Data:   0%|          | 0/100 [00:00<?, ?it/s]

Number of MpCN Samples Now Available: 500099


In [47]:
#Experiment 6: Set-up

#This experiment is based on Model Problem B

#We Specifying Problem Parameters as follows
#Model dimension and parameter space size are
ModDmEx5 = 7
NumParmsEx5 = int(ModDmEx5*(ModDmEx5 -1)/2)


#Pot(A) = kap^{-1}||(A - kap I)^{-1} A||^2
kapEx5 = 0.01 #Make kap smaller to get a stiffer for A oriented problem problem
sigEx5 = 0.01 #Make sigma smaller for a likelihood dominated problem
cov0 = 9


gam = 1.1
CovDiag = [cov0* (j**(-gam)) for j in list(range(1,NumParmsEx5+1))]
#Q = MCMCsmp.rndm_orth_matrix(NumParmsEx4)
#CovEx4 = Q.T@ MCMCsmp.mkDiagCov(CovDiag)@ Q
CovEx5 = MCMCsmp.mkDiagCov(CovDiag)

def FindMatchFull(numParm, modDm, datadim, gdim, Cov, kap, tol, lam):
    curErr = 2
    IddGo = np.diag(np.concatenate([np.ones(gdim), np.zeros(modDm-gdim)]))
    mnA = np.zeros(numParm)
    mnGo = np.zeros(modDm)
    CovInv = np.linalg.inv(Cov)
    numtries = 1
    while curErr > tol:
        Acur = np.random.multivariate_normal(mnA, Cov,1)[0]
        Gcur = np.random.multivariate_normal(mnGo, IddGo,1)[0]
        obsDircur = np.random.multivariate_normal(np.zeros(modDm), np.diag(np.ones(modDm)),datadim)
        Qcur = MCMCsmp.rndm_orth_matrix(numParm)
        tAcur = lam* Qcur @ Acur @ Qcur
        thAdiffcur = MCMCsmp.getThA(modDm, Acur, Gcur, kap) - MCMCsmp.getThA(modDm, tAcur, Gcur, kap)
        curAerr = np.linalg.norm(np.array([v @ thAdiffcur for v in obsDircur])) 
        rotInBulk = max(0,np.dot(CovInv @ tAcur, tAcur) -np.dot(CovInv @ Acur, Acur))
        curErr = min(curAerr + rotInBulk,curErr)
        numtries = numtries+ 1
        if numtries % 100000 == 0:
            print("Indx: " + str(numtries))
            print("A error: " +str(curAerr))
            print("Expansion: "+ str(rotInBulk))
            print("Running Error" + str(curErr))
    return Acur, Gcur, tAcur,obsDircur, numtries


    thA = getThA(ModDm, A, gvec, kap)
    thAProj = np.array([v @ thA for v in obsdir])

datacomp2nd4th = np.array([1,1,0,0,0,0,0])

Atru, gtru, tA,obsDir, n= FindMatchFull(NumParmsEx5, ModDmEx5, 2,2, CovEx5, kapEx5, .00005, .9)

print(Atru)
print(gtru)
print(tA)
print(obsDir)
print(n)

print(np.array([v @ MCMCsmp.getThA(ModDmEx5, Atru, gtru, kapEx5) for v in obsDir]))
print(np.array([v @ MCMCsmp.getThA(ModDmEx5, tA, gtru, kapEx5) for v in obsDir]))


[-2.31523756 -3.47479592  0.43477561  1.05704938 -0.76506519 -0.56007374
  0.04305228 -1.04812716  1.31412553 -0.57192851  1.43020053 -0.13038176
 -0.45947084 -0.78031158  0.23456269  0.15935545 -0.20855903 -0.64102968
  0.51369985 -0.76806932  0.98037976]
[-0.20015761  0.05627247  0.          0.          0.          0.
  0.        ]
[-2.0837138  -3.12731633  0.39129805  0.95134444 -0.68855867 -0.50406636
  0.03874706 -0.94331444  1.18271297 -0.51473566  1.28718048 -0.11734359
 -0.41352375 -0.70228042  0.21110642  0.14341991 -0.18770313 -0.57692672
  0.46232987 -0.69126239  0.88234178]
[[ 2.72929017e-01 -9.87586482e-01  5.93746689e-01 -1.05399464e-03
  -7.62574281e-01  8.16748704e-01  2.88444101e-01]
 [ 1.52929220e+00  1.18503921e+00  4.36656325e-01 -6.94569300e-01
  -1.32383606e+00  5.52006555e-01  1.92736636e+00]]
89929
[-0.73613301  1.40970973]
[-0.73614115  1.40974127]


In [49]:
#Experiment 6: Set-up

#This experiment is based on Model Problem B

#We Specifying Problem Parameters as follows
#Model dimension and parameter space size are
ModDmEx5 = 7
NumParmsEx5 = int(ModDmEx5*(ModDmEx5 -1)/2)

#This time we set g = [1,...,1] and z = kap^{-1} g which is
#leads to a stiff log likihood of the form 
#Pot(A) = kap^{-1}||(A - kap I)^{-1} A||^2
kapEx5 = 0.01 #Make kap smaller to get a stiffer for A oriented problem problem
sigEx5 = 0.01 #Make sigma smaller for a likelihood dominated problem
cov0 = 9


gam = 1.1
CovDiag = [cov0* (j**(-gam)) for j in list(range(1,NumParmsEx5+1))]
#Q = MCMCsmp.rndm_orth_matrix(NumParmsEx4)
#CovEx4 = Q.T@ MCMCsmp.mkDiagCov(CovDiag)@ Q
CovEx5 = MCMCsmp.mkDiagCov(CovDiag)



gvecEx5 = [-0.20015761, 0.05627247, 0, 0, 0, 0, 0]



#print(MCMCsmp.getThA(ModDmEx5, Atru, gtru, kapEx5)*datacomp2nd4th)
#print(MCMCsmp.getThA(ModDmEx5, tA, gtru, kapEx5)*datacomp2nd4th)
#print(MCMCsmp.getThA(ModDmEx5, Atru, gvecEx5, kapEx5)*datacomp2nd4th)
#print(MCMCsmp.getThA(ModDmEx5, tA, gvecEx5, kapEx5)*datacomp2nd4th)


zEx5 = np.array([v @ MCMCsmp.getThA(ModDmEx5, Atru, gtru, kapEx5) for v in obsDir])
print(zEx5)



expId =  19 #Experiment ID
FileNmBase = "Data/Large_p_study/Experiment_5/" + "Ex_ID_"+ str(expId) + "/"
os.makedirs(FileNmBase, exist_ok=True)

probDataFile = FileNmBase + "Problem_Info.txt"
with open(probDataFile, 'w') as file:
    file.write("Target Type:  Model Problem B \n")
    file.write("Model Dim: " + str(NumParmsEx5) + "\n")
    file.write("g: " + str(gvecEx5) + "\n")
    file.write("z: " + str(zEx5) + "\n")
    file.write("sig: " + str(sigEx5)+ "\n")
    file.write("kappa: " + str(kapEx5)+ "\n")
    file.write("Prior Cov: " + str(CovEx5) + "\n")

#Make Loglikehood function with lambda workaround
#using PotExAD(a, gvec, sig, ModDm, z, kap, dataDim, mode = None):
#print(inspect.signature(MCMCsmp.PotExAD))

PotEx5Pass = partial(MCMCsmp.PotExAD_proj, gvec=gvecEx5, sig=sigEx5, ModDm=ModDmEx5, z=zEx5, kap=kapEx5, obsdir=obsDir,  mode = "soft")

[-0.73613301  1.40970973]


AttributeError: module 'MCMC_Sampliers' has no attribute 'PotExAD_proj'